# Network Intrusion Detection System (NIDS) - CIC-IDS2017
Notebook này tổng hợp toàn bộ quy trình từ Tiền xử lý dữ liệu, Cân bằng dữ liệu, Huấn luyện 5 mô hình và So sánh kết quả.

## 1. Tiền xử lý dữ liệu (Preprocessing)
Thực hiện load dữ liệu, làm sạch và xử lý các giá trị vô hạn/thiếu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import joblib
from src.preprocess import run_preprocessing

# Chạy preprocessing (nếu chưa có file cleaned)
if not os.path.exists('outputs/cicids2017_cleaned.csv'):
    run_preprocessing()
else:
    print("Đã có file cleaned.csv, bỏ qua bước này.")

## 2. Cân bằng dữ liệu & Chọn đặc trưng (Balancing & Feature Selection)
Sử dụng SMOTE và UnderSampling để giải quyết vấn đề mất cân bằng lớp.

In [ ]:
from src.balance_and_select import run_balancing

# Chạy balancing (nếu chưa có file balanced)
if not os.path.exists('outputs/cicids2017_balanced.csv'):
    run_balancing()
else:
    print("Đã có file balanced.csv, bỏ qua bước này.")

## 3. Chuẩn bị dữ liệu cho huấn luyện
Load dữ liệu đã cân bằng và chia tập Train/Test.

In [ ]:
df = pd.read_csv('outputs/cicids2017_balanced.csv')
le = joblib.load('models/label_encoder.pkl')
scaler = joblib.load('models/scaler.pkl')

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

X_train_scaled = train_df.drop(['Label', 'split'], axis=1)
y_train_bal = train_df['Label']
X_test_scaled = test_df.drop(['Label', 'split'], axis=1)
y_test = test_df['Label']

selected_features = X_train_scaled.columns.tolist()
print(f"Sẵn sàng với {len(selected_features)} đặc trưng.")

## 4. Huấn luyện các mô hình

In [ ]:
# ── CELL 14 — KNN ──
from src.knn_model import train_knn
knn_model, y_pred_knn = train_knn(X_train_scaled, y_train_bal, X_test_scaled, y_test, le)

In [ ]:
# ── CELL 15 — Random Forest ──
from src.random_forest_model_train import train_random_forest
rf_model, y_pred_rf = train_random_forest(X_train_scaled, y_train_bal, X_test_scaled, y_test, le)

In [ ]:
# ── CELL 16 — Logistic Regression ──
from src.logistic_regression_model import train_logistic_regression
lr_model, y_pred_lr = train_logistic_regression(X_train_scaled, y_train_bal, X_test_scaled, y_test, le)

In [ ]:
# ── CELL 17 — SVM (LinearSVC) ──
from src.svm_model import train_svm
svm_model, y_pred_svm = train_svm(X_train_scaled, y_train_bal, X_test_scaled, y_test, le)

In [ ]:
# ── CELL 18 — Naive Bayes ──
from src.naive_bayes_model import train_naive_bayes
nb_model, y_pred_nb = train_naive_bayes(X_train_scaled, y_train_bal, X_test_scaled, y_test, le)

## 5. So sánh kết quả & Mô phỏng

In [ ]:
# ── CELL 19 — Tổng hợp bảng so sánh 5 model ──
from src.random_forest_model_train import build_comparison_table
df_cmp = build_comparison_table(y_test, y_pred_lr, y_pred_svm, y_pred_nb, y_pred_knn, y_pred_rf)

In [ ]:
# ── CELL 20 — Real-time alert simulation ──
from src.deploy_realtime import run_simulation
run_simulation(X_test_scaled.values, selected_features, rf_model, scaler, le, n_samples=15)